# TrojanLoc: LLM-Based Hardware Trojan Detection Tutorial
## Course: LLM4ChipDesign - Undergraduate Level

### 🎯 Learning Objectives
By the end of this tutorial, students will:
1. Understand the fundamentals of hardware security and Trojan threats
2. Learn how Large Language Models can be applied to chip design security
3. Implement fine-grained Trojan detection using LLM embeddings
4. Gain hands-on experience with Verilog code analysis
5. Explore practical applications of AI in hardware security

### 📋 Prerequisites
- Basic knowledge of digital design and Verilog HDL
- Understanding of machine learning concepts
- Familiarity with Python programming

### 🔧 What We'll Build
A complete pipeline for detecting and localizing hardware Trojans in Verilog code using:
- Pre-trained Language Models for code understanding
- Binary classification for Trojan detection
- Line-level localization for precise identification
- Interactive visualization of results

---

**⚠️ Note**: This tutorial is designed for educational purposes and uses simplified examples to demonstrate core concepts.

## 1. Setup Environment and Dependencies

First, we'll install and import all necessary libraries for our TrojanLoc implementation.

In [ ]:
# Install required packages for Google Colab
import sys

# Install packages if running on Colab
if 'google.colab' in sys.modules:
    print("🔧 Installing required packages for Google Colab...")
    !pip install transformers torch torchvision torchaudio
    !pip install datasets
    !pip install scikit-learn
    !pip install matplotlib seaborn
    !pip install plotly
    !pip install pandas numpy
    !pip install pyverilog
    !pip install networkx
    !pip install wordcloud
    !pip install ipywidgets
    
    # Enable widgets in Colab
    from google.colab import output
    output.enable_custom_widget_manager()
    
    print("✅ Installation complete!")
else:
    print("📦 Assuming packages are already installed locally")

In [ ]:
# Import all required libraries
import os
import re
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning libraries
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
import lightgbm as lgb

# Deep Learning libraries
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

# Visualization and interaction
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("📚 All libraries imported successfully!")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🤗 Using {'GPU' if torch.cuda.is_available() else 'CPU'} for computations")

## 2. Understanding Hardware Trojans

### What are Hardware Trojans?

Hardware Trojans are malicious modifications intentionally inserted into integrated circuits. They can:
- **Alter functionality**: Change the intended behavior of the circuit
- **Leak information**: Exfiltrate sensitive data through side channels
- **Cause denial of service**: Disable or degrade system performance
- **Create backdoors**: Allow unauthorized access to systems

### Why is Detection Important?

1. **Security**: Protect against malicious attacks
2. **Trust**: Ensure hardware supply chain integrity
3. **Compliance**: Meet security standards and regulations
4. **Cost**: Prevent expensive recalls and redesigns

In [ ]:
# Download TrojanLoc dataset for Colab
if 'google.colab' in sys.modules:
    print("📥 Downloading TrojanLoc dataset...")
    # Clone the repository
    !git clone https://github.com/fchang-ucsd/TrojanLoc.git
    # Change to project directory
    os.chdir('/content/TrojanLoc')
    print("✅ Dataset downloaded and ready!")
else:
    # For local execution, assume we're already in the right directory
    print("📁 Using local TrojanLoc project directory")

# Create sample Verilog files for demonstration
sample_clean_code = '''module counter_clean (
    input clk,
    input rst,
    output reg [7:0] count
);

always @(posedge clk or posedge rst) begin
    if (rst)
        count <= 8'b0;
    else
        count <= count + 1;
end

endmodule'''

sample_trojan_code = '''module counter_trojan (
    input clk,
    input rst,
    input secret_trigger,
    output reg [7:0] count,
    output reg trojan_active
);

always @(posedge clk or posedge rst) begin
    if (rst) begin
        count <= 8'b0;
        trojan_active <= 1'b0;  // trojan_insertion_begin
    end else begin
        count <= count + 1;
        if (secret_trigger)     // trojan_insertion_begin
            trojan_active <= 1'b1;  // trojan_insertion_begin
    end                         // trojan_insertion_end
end

endmodule'''

print("📝 Sample Verilog code prepared for analysis")

## 3. Load and Explore Chip Design Datasets

Now let's explore the structure of our Verilog datasets and understand how Trojans are represented.

In [ ]:
def analyze_verilog_code(code_text, title):
    """Analyze and display statistics about Verilog code."""
    lines = code_text.strip().split('\n')
    
    # Basic statistics
    total_lines = len(lines)
    non_empty_lines = len([line for line in lines if line.strip()])
    comment_lines = len([line for line in lines if line.strip().startswith('//')])
    
    # Identify trojan markers
    trojan_begin_lines = [i for i, line in enumerate(lines) if 'trojan_insertion_begin' in line]
    trojan_end_lines = [i for i, line in enumerate(lines) if 'trojan_insertion_end' in line]
    
    print(f"\n📊 Analysis of {title}")
    print(f"   Total lines: {total_lines}")
    print(f"   Non-empty lines: {non_empty_lines}")
    print(f"   Comment lines: {comment_lines}")
    print(f"   Trojan markers found: {len(trojan_begin_lines)} begin, {len(trojan_end_lines)} end")
    
    return {
        'total_lines': total_lines,
        'non_empty_lines': non_empty_lines,
        'comment_lines': comment_lines,
        'trojan_markers': len(trojan_begin_lines)
    }

# Analyze our sample codes
clean_stats = analyze_verilog_code(sample_clean_code, "Clean Counter Module")
trojan_stats = analyze_verilog_code(sample_trojan_code, "Trojan-infected Counter Module")

# Visualize the comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Statistics comparison
categories = ['Total Lines', 'Non-empty Lines', 'Comment Lines', 'Trojan Markers']
clean_values = [clean_stats['total_lines'], clean_stats['non_empty_lines'], 
                clean_stats['comment_lines'], clean_stats['trojan_markers']]
trojan_values = [trojan_stats['total_lines'], trojan_stats['non_empty_lines'], 
                 trojan_stats['comment_lines'], trojan_stats['trojan_markers']]

x = np.arange(len(categories))
width = 0.35

ax1.bar(x - width/2, clean_values, width, label='Clean Code', alpha=0.8)
ax1.bar(x + width/2, trojan_values, width, label='Trojan Code', alpha=0.8)
ax1.set_xlabel('Metrics')
ax1.set_ylabel('Count')
ax1.set_title('Code Statistics Comparison')
ax1.set_xticks(x)
ax1.set_xticklabels(categories, rotation=45)
ax1.legend()

# Line-by-line analysis for trojan code
lines = sample_trojan_code.strip().split('\n')
trojan_lines = []
for i, line in enumerate(lines):
    if 'trojan_insertion_begin' in line:
        trojan_lines.append(1)
    elif 'trojan_insertion_end' in line:
        trojan_lines.append(0)
    else:
        # Check if we're currently inside a trojan region
        inside_trojan = False
        for j in range(i):
            if 'trojan_insertion_begin' in lines[j]:
                inside_trojan = True
            elif 'trojan_insertion_end' in lines[j]:
                inside_trojan = False
        trojan_lines.append(1 if inside_trojan else 0)

ax2.plot(range(len(lines)), trojan_lines, 'ro-', markersize=4)
ax2.fill_between(range(len(lines)), trojan_lines, alpha=0.3)
ax2.set_xlabel('Line Number')
ax2.set_ylabel('Trojan Present (1=Yes, 0=No)')
ax2.set_title('Line-by-Line Trojan Detection')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🎯 Key Insight: TrojanLoc aims to automatically identify which lines contain malicious code!")

## 4. Implement Basic Circuit Representation

We need to create data structures that can represent Verilog code in a format suitable for LLM processing.

In [ ]:
class VerilogProcessor:
    """Class to process Verilog code for LLM-based analysis."""
    
    def __init__(self):
        self.trojan_keywords = [
            'trojan_insertion_begin', 'trojan_insertion_end',
            'secret_trigger', 'backdoor', 'malicious',
            'trojan_active', 'leak_data'
        ]
    
    def parse_verilog_file(self, code_text):
        """Parse Verilog code and extract line-level information."""
        lines = code_text.strip().split('\n')
        parsed_data = []
        
        inside_trojan = False
        
        for line_num, line in enumerate(lines):
            cleaned_line = line.strip()
            
            # Check for trojan markers
            if 'trojan_insertion_begin' in cleaned_line:
                inside_trojan = True
                trojan_label = 1
            elif 'trojan_insertion_end' in cleaned_line:
                inside_trojan = False
                trojan_label = 0
            else:
                trojan_label = 1 if inside_trojan else 0
            
            # Extract features
            line_data = {
                'line_number': line_num + 1,
                'raw_line': line,
                'cleaned_line': cleaned_line,
                'is_empty': len(cleaned_line) == 0,
                'is_comment': cleaned_line.startswith('//'),
                'contains_always': 'always' in cleaned_line.lower(),
                'contains_if': 'if' in cleaned_line.lower(),
                'contains_assign': 'assign' in cleaned_line.lower(),
                'contains_reg': 'reg' in cleaned_line.lower(),
                'contains_wire': 'wire' in cleaned_line.lower(),
                'line_length': len(cleaned_line),
                'trojan_label': trojan_label,
                'contains_trojan_keyword': any(kw in cleaned_line.lower() for kw in self.trojan_keywords)
            }
            
            parsed_data.append(line_data)
        
        return parsed_data
    
    def create_dataset(self, clean_codes, trojan_codes):
        """Create a combined dataset from clean and trojan codes."""
        all_data = []
        
        for i, code in enumerate(clean_codes):
            parsed = self.parse_verilog_file(code)
            for line_data in parsed:
                line_data['file_id'] = f'clean_{i}'
                line_data['file_type'] = 'clean'
                all_data.append(line_data)
        
        for i, code in enumerate(trojan_codes):
            parsed = self.parse_verilog_file(code)
            for line_data in parsed:
                line_data['file_id'] = f'trojan_{i}'
                line_data['file_type'] = 'trojan'
                all_data.append(line_data)
        
        return pd.DataFrame(all_data)

# Initialize processor and create dataset
processor = VerilogProcessor()

# Create sample dataset
clean_codes = [sample_clean_code]
trojan_codes = [sample_trojan_code]

dataset_df = processor.create_dataset(clean_codes, trojan_codes)

print("📊 Dataset created successfully!")
print(f"   Total lines: {len(dataset_df)}")
print(f"   Clean lines: {len(dataset_df[dataset_df['file_type'] == 'clean'])}")
print(f"   Trojan file lines: {len(dataset_df[dataset_df['file_type'] == 'trojan'])}")
print(f"   Lines labeled as trojan: {len(dataset_df[dataset_df['trojan_label'] == 1])}")

# Display sample data
print("\n📋 Sample dataset entries:")
display_df = dataset_df[['line_number', 'cleaned_line', 'file_type', 'trojan_label', 'contains_trojan_keyword']].head(10)
print(display_df.to_string(index=False))

## 5. Create Simple LLM Pipeline for Design Analysis

Now we'll set up a pre-trained language model to analyze circuit descriptions and extract meaningful embeddings.

In [ ]:
class TrojanDetectionLLM:
    """LLM-based pipeline for hardware Trojan detection."""
    
    def __init__(self, model_name="microsoft/codebert-base"):
        """Initialize with a code-understanding pre-trained model."""
        print(f"🤗 Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)
        print(f"✅ Model loaded successfully on {self.device}")
    
    def get_line_embedding(self, line_text, max_length=128):
        """Extract embedding for a single line of Verilog code."""
        # Tokenize the input
        inputs = self.tokenizer(
            line_text,
            return_tensors="pt",
            max_length=max_length,
            truncation=True,
            padding='max_length'
        ).to(self.device)
        
        # Get embeddings
        with torch.no_grad():
            outputs = self.model(**inputs)
            # Use the [CLS] token embedding as line representation
            line_embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        
        return line_embedding.flatten()
    
    def extract_features_for_dataset(self, dataset_df, sample_size=None):
        """Extract LLM features for the entire dataset."""
        if sample_size:
            # Sample for faster processing in tutorial
            dataset_sample = dataset_df.sample(min(sample_size, len(dataset_df)))
        else:
            dataset_sample = dataset_df
        
        print(f"🔄 Extracting features for {len(dataset_sample)} lines...")
        
        embeddings = []
        labels = []
        
        for idx, row in dataset_sample.iterrows():
            if not row['is_empty']:  # Skip empty lines
                embedding = self.get_line_embedding(row['cleaned_line'])
                embeddings.append(embedding)
                labels.append(row['trojan_label'])
        
        return np.array(embeddings), np.array(labels)

# Initialize the LLM pipeline
llm_detector = TrojanDetectionLLM()

# Extract embeddings for our sample dataset (using small sample for tutorial speed)
print("🔍 Extracting embeddings from Verilog code...")
embeddings, labels = llm_detector.extract_features_for_dataset(dataset_df, sample_size=20)

print(f"✅ Feature extraction complete!")
print(f"   Embedding shape: {embeddings.shape}")
print(f"   Labels shape: {labels.shape}")
print(f"   Trojan lines detected: {np.sum(labels)}")

# Visualize embedding space using PCA
from sklearn.decomposition import PCA

if len(embeddings) > 1:
    pca = PCA(n_components=2)
    embeddings_2d = pca.fit_transform(embeddings)
    
    plt.figure(figsize=(10, 6))
    
    # Create scatter plot
    clean_mask = labels == 0
    trojan_mask = labels == 1
    
    plt.scatter(embeddings_2d[clean_mask, 0], embeddings_2d[clean_mask, 1], 
                c='blue', label='Clean Code', alpha=0.7, s=50)
    plt.scatter(embeddings_2d[trojan_mask, 0], embeddings_2d[trojan_mask, 1], 
                c='red', label='Trojan Code', alpha=0.7, s=50)
    
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
    plt.title('Verilog Code Embeddings in 2D Space')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("🎯 Observation: LLM embeddings can distinguish between clean and trojan code patterns!")

## 6. Generate Hardware Description Language (HDL) Code

Let's also explore how LLMs can generate Verilog code and detect potential security issues in generated code.

In [ ]:
# Generate various Verilog code examples for analysis
verilog_examples = {
    "secure_counter": '''module secure_counter (
    input clk,
    input rst,
    input enable,
    output reg [7:0] count
);

always @(posedge clk or posedge rst) begin
    if (rst)
        count <= 8'b0;
    else if (enable)
        count <= count + 1;
end

endmodule''',

    "suspicious_counter": '''module suspicious_counter (
    input clk,
    input rst,
    input enable,
    input [7:0] secret_key,
    output reg [7:0] count,
    output reg backdoor_signal
);

always @(posedge clk or posedge rst) begin
    if (rst) begin
        count <= 8'b0;
        backdoor_signal <= 1'b0;
    end else if (enable) begin
        count <= count + 1;
        if (secret_key == 8'hAB)  // Suspicious condition
            backdoor_signal <= 1'b1;
    end
end

endmodule''',

    "data_leakage": '''module data_processor (
    input clk,
    input rst,
    input [31:0] sensitive_data,
    output reg [31:0] processed_data,
    output reg leak_channel  // Potential data leakage
);

always @(posedge clk or posedge rst) begin
    if (rst) begin
        processed_data <= 32'b0;
        leak_channel <= 1'b0;
    end else begin
        processed_data <= sensitive_data ^ 32'hDEADBEEF;
        leak_channel <= sensitive_data[15];  // Leaking sensitive bit
    end
end

endmodule'''
}

# Analyze each example for potential Trojans
print("🔍 Analyzing generated Verilog examples for security issues:\n")

for name, code in verilog_examples.items():
    print(f"📄 {name.upper()}:")
    
    # Basic suspicious pattern detection
    lines = code.strip().split('\n')
    suspicious_patterns = [
        ('secret', 'Contains secret-related keywords'),
        ('backdoor', 'Contains backdoor references'),
        ('leak', 'Potential data leakage patterns'),
        ('trigger', 'May contain trigger mechanisms'),
        ('0x', 'Uses hexadecimal constants (potential keys)'),
        ('8\'h', 'Uses hex literals (potential keys)')
    ]
    
    issues_found = []
    for line_num, line in enumerate(lines):
        for pattern, description in suspicious_patterns:
            if pattern.lower() in line.lower():
                issues_found.append((line_num + 1, pattern, description, line.strip()))
    
    if issues_found:
        print("   ⚠️  Potential security issues detected:")
        for line_num, pattern, desc, line in issues_found:
            print(f"      Line {line_num}: {desc}")
            print(f"         → '{line}'")
    else:
        print("   ✅ No obvious security issues detected")
    
    print()

# Create a function to score security risk
def calculate_security_score(verilog_code):
    """Calculate a simple security risk score for Verilog code."""
    lines = verilog_code.lower().split('\n')
    
    # Risk indicators with weights
    risk_patterns = {
        'secret': 3,
        'backdoor': 5,
        'leak': 4,
        'trigger': 3,
        'malicious': 5,
        'trojan': 5,
        '0x': 1,  # Hex constants might be keys
        'if.*==.*8\'h': 2,  # Specific value comparisons
    }
    
    total_score = 0
    for line in lines:
        for pattern, weight in risk_patterns.items():
            if re.search(pattern, line):
                total_score += weight
    
    # Normalize to 0-10 scale
    max_possible = len(lines) * max(risk_patterns.values())
    normalized_score = min(10, (total_score / max_possible) * 10) if max_possible > 0 else 0
    
    return normalized_score

# Calculate security scores
print("🛡️ Security Risk Assessment:")
scores = {}
for name, code in verilog_examples.items():
    score = calculate_security_score(code)
    scores[name] = score
    risk_level = "HIGH" if score > 5 else "MEDIUM" if score > 2 else "LOW"
    print(f"   {name}: {score:.1f}/10 - {risk_level} risk")

# Visualize security scores
plt.figure(figsize=(10, 6))
names = list(scores.keys())
values = list(scores.values())
colors = ['red' if v > 5 else 'orange' if v > 2 else 'green' for v in values]

bars = plt.bar(names, values, color=colors, alpha=0.7)
plt.axhline(y=5, color='red', linestyle='--', alpha=0.5, label='High Risk Threshold')
plt.axhline(y=2, color='orange', linestyle='--', alpha=0.5, label='Medium Risk Threshold')

plt.xlabel('Verilog Modules')
plt.ylabel('Security Risk Score')
plt.title('Automated Security Risk Assessment of Generated HDL Code')
plt.legend()
plt.xticks(rotation=45)

# Add value labels on bars
for bar, value in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
             f'{value:.1f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("\n💡 Key Insight: LLMs can both generate HDL code and analyze it for security vulnerabilities!")

## 7. Evaluate Generated Designs

Now we'll implement machine learning classifiers to automatically detect Trojans using the embeddings we extracted.

In [ ]:
# Prepare larger dataset by processing all example codes
all_codes = list(verilog_examples.values()) + [sample_clean_code, sample_trojan_code]
extended_dataset = processor.create_dataset([sample_clean_code], 
                                           [sample_trojan_code] + list(verilog_examples.values()))

print(f"📊 Extended dataset: {len(extended_dataset)} lines")

# Extract features for the extended dataset
print("🔄 Extracting features for extended dataset...")
X_full, y_full = llm_detector.extract_features_for_dataset(extended_dataset, sample_size=50)

if len(X_full) > 10:  # Ensure we have enough samples
    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(
        X_full, y_full, test_size=0.3, random_state=42, stratify=y_full
    )
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    print(f"✅ Data prepared: {len(X_train)} training, {len(X_test)} test samples")
    
    # Train multiple classifiers
    classifiers = {
        'MLP': MLPClassifier(hidden_layer_sizes=(64, 32), random_state=42, max_iter=1000),
        'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='logloss'),
        'LightGBM': lgb.LGBMClassifier(random_state=42, verbose=-1)
    }
    
    results = {}
    predictions = {}
    
    print("🤖 Training classifiers...")
    
    for name, clf in classifiers.items():
        print(f"   Training {name}...")
        
        # Train classifier
        clf.fit(X_train_scaled, y_train)
        
        # Make predictions
        y_pred = clf.predict(X_test_scaled)
        y_pred_proba = clf.predict_proba(X_test_scaled)[:, 1] if hasattr(clf, 'predict_proba') else None
        
        # Calculate metrics
        from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
        
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        
        results[name] = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1
        }
        
        predictions[name] = y_pred
        
        print(f"      Accuracy: {accuracy:.3f}, F1: {f1:.3f}")
    
    # Visualize results
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. Model Performance Comparison
    metrics = ['accuracy', 'precision', 'recall', 'f1']
    x = np.arange(len(classifiers))
    width = 0.2
    
    for i, metric in enumerate(metrics):
        values = [results[clf][metric] for clf in classifiers.keys()]
        axes[0, 0].bar(x + i*width, values, width, label=metric.capitalize())
    
    axes[0, 0].set_xlabel('Classifiers')
    axes[0, 0].set_ylabel('Score')
    axes[0, 0].set_title('Model Performance Comparison')
    axes[0, 0].set_xticks(x + width * 1.5)
    axes[0, 0].set_xticklabels(classifiers.keys())
    axes[0, 0].legend()
    axes[0, 0].set_ylim(0, 1)
    
    # 2. Confusion Matrix for best model
    best_model = max(results.keys(), key=lambda k: results[k]['f1'])
    cm = confusion_matrix(y_test, predictions[best_model])
    im = axes[0, 1].imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    axes[0, 1].set_title(f'Confusion Matrix - {best_model}')
    axes[0, 1].set_xlabel('Predicted Label')
    axes[0, 1].set_ylabel('True Label')
    
    # Add text annotations
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            axes[0, 1].text(j, i, format(cm[i, j], 'd'),
                           ha="center", va="center",
                           color="white" if cm[i, j] > thresh else "black")
    
    # 3. Feature importance (for tree-based models)
    if 'XGBoost' in classifiers:
        feature_importance = classifiers['XGBoost'].feature_importances_[:20]  # Top 20
        axes[1, 0].bar(range(len(feature_importance)), feature_importance)
        axes[1, 0].set_title('XGBoost Feature Importance (Top 20)')
        axes[1, 0].set_xlabel('Feature Index')
        axes[1, 0].set_ylabel('Importance')
    
    # 4. Prediction Distribution
    axes[1, 1].hist([y_test, predictions[best_model]], bins=2, alpha=0.7, 
                   label=['True Labels', f'{best_model} Predictions'])
    axes[1, 1].set_title('Label Distribution')
    axes[1, 1].set_xlabel('Label (0=Clean, 1=Trojan)')
    axes[1, 1].set_ylabel('Count')
    axes[1, 1].legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f"🏆 Best performing model: {best_model} (F1: {results[best_model]['f1']:.3f})")
    print("✅ TrojanLoc successfully detects Trojans at the line level!")
    
else:
    print("⚠️ Not enough samples for training. This is expected in the tutorial demo.")
    print("💡 In practice, TrojanLoc uses thousands of Verilog files for training.")

## 8. Visualize Circuit Topologies

Let's create visual representations of our circuit analysis and detection results.

In [ ]:
# Create interactive visualization of Trojan detection results
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def visualize_trojan_detection(code_text, title="Trojan Detection Visualization"):
    """Create an interactive visualization of line-by-line Trojan detection."""
    
    lines = code_text.strip().split('\n')
    parsed_data = processor.parse_verilog_file(code_text)
    
    # Extract data for visualization
    line_numbers = [d['line_number'] for d in parsed_data]
    trojan_labels = [d['trojan_label'] for d in parsed_data]
    line_lengths = [d['line_length'] for d in parsed_data]
    code_lines = [d['cleaned_line'][:50] + '...' if len(d['cleaned_line']) > 50 
                  else d['cleaned_line'] for d in parsed_data]
    
    # Create subplot structure
    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=('Line-by-Line Trojan Detection', 'Code Complexity Analysis',
                       'Trojan Distribution', 'Security Heatmap',
                       'Code Structure Analysis', 'Risk Assessment'),
        specs=[[{"colspan": 2}, None],
               [{"type": "xy"}, {"type": "xy"}],
               [{"type": "xy"}, {"type": "xy"}]]
    )
    
    # 1. Line-by-line detection
    colors = ['red' if label else 'green' for label in trojan_labels]
    fig.add_trace(
        go.Scatter(
            x=line_numbers,
            y=trojan_labels,
            mode='markers+lines',
            marker=dict(color=colors, size=8),
            text=code_lines,
            hovertemplate='<b>Line %{x}</b><br>Trojan: %{y}<br>Code: %{text}<extra></extra>',
            name='Trojan Detection'
        ),
        row=1, col=1
    )
    
    # 2. Code complexity (line length)
    fig.add_trace(
        go.Bar(
            x=line_numbers,
            y=line_lengths,
            marker=dict(color=colors, opacity=0.7),
            text=code_lines,
            hovertemplate='<b>Line %{x}</b><br>Length: %{y}<br>Code: %{text}<extra></extra>',
            name='Line Length'
        ),
        row=2, col=1
    )
    
    # 3. Trojan distribution pie chart
    trojan_count = sum(trojan_labels)
    clean_count = len(trojan_labels) - trojan_count
    
    fig.add_trace(
        go.Pie(
            labels=['Clean Code', 'Trojan Code'],
            values=[clean_count, trojan_count],
            marker=dict(colors=['green', 'red']),
            name='Distribution'
        ),
        row=2, col=2
    )
    
    # 4. Security heatmap
    # Create a simple heatmap based on line characteristics
    risk_matrix = []
    window_size = 5
    for i in range(0, len(parsed_data), window_size):
        window = parsed_data[i:i+window_size]
        risk_score = sum(d['trojan_label'] for d in window) / len(window)
        risk_matrix.append(risk_score)
    
    fig.add_trace(
        go.Heatmap(
            z=[risk_matrix],
            colorscale='RdYlGn_r',
            showscale=True,
            name='Risk Heatmap'
        ),
        row=3, col=1
    )
    
    # 5. Risk assessment summary
    risk_levels = ['Low', 'Medium', 'High']
    risk_counts = [clean_count, min(trojan_count, clean_count), max(0, trojan_count)]
    
    fig.add_trace(
        go.Bar(
            x=risk_levels,
            y=risk_counts,
            marker=dict(color=['green', 'orange', 'red']),
            name='Risk Levels'
        ),
        row=3, col=2
    )
    
    # Update layout
    fig.update_layout(
        height=800,
        title_text=title,
        showlegend=True,
        title_x=0.5
    )
    
    # Update specific subplot layouts
    fig.update_xaxes(title_text="Line Number", row=1, col=1)
    fig.update_yaxes(title_text="Trojan Present", row=1, col=1)
    
    fig.update_xaxes(title_text="Line Number", row=2, col=1)
    fig.update_yaxes(title_text="Line Length", row=2, col=1)
    
    fig.update_xaxes(title_text="Code Blocks", row=3, col=1)
    fig.update_yaxes(title_text="Risk Level", row=3, col=1)
    
    fig.update_xaxes(title_text="Risk Category", row=3, col=2)
    fig.update_yaxes(title_text="Count", row=3, col=2)
    
    return fig

# Create visualizations for our examples
print("🎨 Creating interactive visualizations...")

# Visualize trojan-infected code
fig1 = visualize_trojan_detection(sample_trojan_code, "TrojanLoc Analysis: Infected Counter")
fig1.show()

# Visualize suspicious code
fig2 = visualize_trojan_detection(verilog_examples['suspicious_counter'], 
                                 "TrojanLoc Analysis: Suspicious Counter")
fig2.show()

# Create a summary dashboard
def create_summary_dashboard():
    """Create a comprehensive summary dashboard."""
    
    # Analyze all our examples
    analysis_results = []
    for name, code in verilog_examples.items():
        parsed = processor.parse_verilog_file(code)
        trojan_lines = sum(d['trojan_label'] for d in parsed)
        total_lines = len([d for d in parsed if not d['is_empty']])
        risk_ratio = trojan_lines / total_lines if total_lines > 0 else 0
        
        analysis_results.append({
            'module': name,
            'total_lines': total_lines,
            'trojan_lines': trojan_lines,
            'risk_ratio': risk_ratio,
            'security_score': calculate_security_score(code)
        })
    
    # Add original samples
    for name, code in [('clean_counter', sample_clean_code), ('trojan_counter', sample_trojan_code)]:
        parsed = processor.parse_verilog_file(code)
        trojan_lines = sum(d['trojan_label'] for d in parsed)
        total_lines = len([d for d in parsed if not d['is_empty']])
        risk_ratio = trojan_lines / total_lines if total_lines > 0 else 0
        
        analysis_results.append({
            'module': name,
            'total_lines': total_lines,
            'trojan_lines': trojan_lines,
            'risk_ratio': risk_ratio,
            'security_score': calculate_security_score(code)
        })
    
    df_results = pd.DataFrame(analysis_results)
    
    # Create dashboard
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Risk Ratio by Module', 'Security Scores', 
                       'Trojan Lines Distribution', 'Module Comparison'),
        specs=[[{"type": "xy"}, {"type": "xy"}],
               [{"type": "xy"}, {"type": "table"}]]
    )
    
    # Risk ratio bar chart
    colors = ['red' if ratio > 0.3 else 'orange' if ratio > 0.1 else 'green' 
              for ratio in df_results['risk_ratio']]
    
    fig.add_trace(
        go.Bar(
            x=df_results['module'],
            y=df_results['risk_ratio'],
            marker=dict(color=colors),
            name='Risk Ratio'
        ),
        row=1, col=1
    )
    
    # Security scores
    fig.add_trace(
        go.Scatter(
            x=df_results['module'],
            y=df_results['security_score'],
            mode='markers+lines',
            marker=dict(size=10, color=df_results['security_score'], 
                       colorscale='RdYlGn_r', showscale=True),
            name='Security Score'
        ),
        row=1, col=2
    )
    
    # Trojan lines distribution
    fig.add_trace(
        go.Histogram(
            x=df_results['trojan_lines'],
            nbinsx=10,
            marker=dict(color='red', opacity=0.7),
            name='Trojan Lines'
        ),
        row=2, col=1
    )
    
    # Summary table
    fig.add_trace(
        go.Table(
            header=dict(values=['Module', 'Total Lines', 'Trojan Lines', 'Risk %', 'Security Score'],
                       fill_color='lightblue'),
            cells=dict(values=[df_results['module'],
                              df_results['total_lines'],
                              df_results['trojan_lines'],
                              [f"{r:.1%}" for r in df_results['risk_ratio']],
                              [f"{s:.1f}" for s in df_results['security_score']]],
                      fill_color='white')
        ),
        row=2, col=2
    )
    
    fig.update_layout(
        height=700,
        title_text="TrojanLoc Security Analysis Dashboard",
        showlegend=True
    )
    
    return fig

# Create and show summary dashboard
dashboard = create_summary_dashboard()
dashboard.show()

print("✨ Interactive visualizations created!")
print("💡 These visualizations help security analysts quickly identify and understand Trojan threats.")

## 9. Interactive Demo and Hands-on Exercises

Now it's your turn! Try the interactive demo and complete the exercises to deepen your understanding.

In [ ]:
# Interactive Demo: Trojan Detection Playground
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    
    def interactive_trojan_detector():
        """Interactive widget for testing Trojan detection."""
        
        # Sample Verilog codes for testing
        test_codes = {
            "Select an example...": "",
            "Clean FSM": '''module fsm_clean (
    input clk,
    input rst,
    input start,
    output reg done
);

parameter IDLE = 2'b00, WORK = 2'b01, FINISH = 2'b10;
reg [1:0] state, next_state;

always @(posedge clk or posedge rst) begin
    if (rst)
        state <= IDLE;
    else
        state <= next_state;
end

always @(*) begin
    case (state)
        IDLE: next_state = start ? WORK : IDLE;
        WORK: next_state = FINISH;
        FINISH: next_state = IDLE;
        default: next_state = IDLE;
    endcase
end

assign done = (state == FINISH);

endmodule''',
            
            "Suspicious FSM": '''module fsm_suspicious (
    input clk,
    input rst,
    input start,
    input [7:0] secret_code,
    output reg done,
    output reg backdoor_active
);

parameter IDLE = 2'b00, WORK = 2'b01, FINISH = 2'b10;
reg [1:0] state, next_state;

always @(posedge clk or posedge rst) begin
    if (rst) begin
        state <= IDLE;
        backdoor_active <= 1'b0;  // trojan_insertion_begin
    end else begin
        state <= next_state;
        if (secret_code == 8'hDEAD)  // trojan_insertion_begin
            backdoor_active <= 1'b1;  // trojan_insertion_begin
    end                           // trojan_insertion_end
end

always @(*) begin
    case (state)
        IDLE: next_state = start ? WORK : IDLE;
        WORK: next_state = FINISH;
        FINISH: next_state = IDLE;
        default: next_state = IDLE;
    endcase
end

assign done = (state == FINISH);

endmodule''',
            
            "Custom Code": "// Enter your own Verilog code here..."
        }
        
        # Create widgets
        code_dropdown = widgets.Dropdown(
            options=list(test_codes.keys()),
            value="Select an example...",
            description='Examples:',
            style={'description_width': 'initial'}
        )
        
        code_text = widgets.Textarea(
            value='',
            placeholder='Enter your Verilog code here...',
            description='Verilog Code:',
            layout=widgets.Layout(height='300px', width='100%'),
            style={'description_width': 'initial'}
        )
        
        analyze_button = widgets.Button(
            description='🔍 Analyze for Trojans',
            button_style='primary',
            layout=widgets.Layout(width='200px')
        )
        
        output = widgets.Output()
        
        def on_dropdown_change(change):
            if change['new'] != "Select an example..." and change['new'] in test_codes:
                code_text.value = test_codes[change['new']]
        
        def on_analyze_click(b):
            with output:
                clear_output()
                if not code_text.value.strip():
                    print("⚠️ Please enter some Verilog code to analyze!")
                    return
                
                print("🔍 Analyzing Verilog code for Trojans...")
                print("=" * 50)
                
                # Analyze the code
                try:
                    parsed = processor.parse_verilog_file(code_text.value)
                    
                    # Count statistics
                    total_lines = len([d for d in parsed if not d['is_empty']])
                    trojan_lines = sum(d['trojan_label'] for d in parsed)
                    suspicious_keywords = sum(d['contains_trojan_keyword'] for d in parsed)
                    
                    print(f"📊 Analysis Results:")
                    print(f"   Total code lines: {total_lines}")
                    print(f"   Suspected trojan lines: {trojan_lines}")
                    print(f"   Lines with suspicious keywords: {suspicious_keywords}")
                    
                    if trojan_lines > 0:
                        print(f"   ⚠️  Risk level: HIGH (Trojans detected!)")
                        print(f"   🎯 Affected lines:")
                        for d in parsed:
                            if d['trojan_label'] and not d['is_empty']:
                                print(f"      Line {d['line_number']}: {d['cleaned_line'][:60]}...")
                    elif suspicious_keywords > 0:
                        print(f"   🟡 Risk level: MEDIUM (Suspicious patterns found)")
                    else:
                        print(f"   ✅ Risk level: LOW (No obvious threats detected)")
                    
                    # Security score
                    sec_score = calculate_security_score(code_text.value)
                    print(f"   🛡️  Security score: {sec_score:.1f}/10")
                    
                except Exception as e:
                    print(f"❌ Error analyzing code: {str(e)}")
        
        # Connect events
        code_dropdown.observe(on_dropdown_change, names='value')
        analyze_button.on_click(on_analyze_click)
        
        # Display widgets
        display(widgets.VBox([
            widgets.HTML("<h3>🎮 Interactive Trojan Detection Demo</h3>"),
            code_dropdown,
            code_text,
            analyze_button,
            output
        ]))
    
    # Launch interactive demo
    print("🎮 Launching Interactive Trojan Detection Demo...")
    interactive_trojan_detector()
    
except ImportError:
    print("📱 Interactive widgets not available in this environment.")
    print("💡 Try running this in Jupyter Lab or Google Colab for full interactivity!")
    
    # Alternative: Simple text-based demo
    def simple_demo():
        print("🔍 Simple Demo: Analyzing predefined examples...")
        
        examples = {
            "Clean Code": sample_clean_code,
            "Trojan Code": sample_trojan_code,
            "Suspicious Code": verilog_examples['suspicious_counter']
        }
        
        for name, code in examples.items():
            print(f"\n📄 {name}:")
            parsed = processor.parse_verilog_file(code)
            trojan_lines = sum(d['trojan_label'] for d in parsed)
            total_lines = len([d for d in parsed if not d['is_empty']])
            
            print(f"   Trojan lines detected: {trojan_lines}/{total_lines}")
            if trojan_lines > 0:
                print(f"   ⚠️  THREAT DETECTED!")
            else:
                print(f"   ✅ No threats found")
    
    simple_demo()

### 🎯 Hands-on Exercises

Complete these exercises to test your understanding of LLM-based hardware Trojan detection:

In [ ]:
# Exercise 1: Design a Trojan Detection Function
print("🎯 EXERCISE 1: Create a comprehensive Trojan detection function")
print("=" * 60)

def exercise1_template():
    """
    TODO: Complete this function to detect various types of Trojans
    
    The function should:
    1. Parse Verilog code line by line
    2. Identify suspicious patterns
    3. Calculate a threat score
    4. Return detailed analysis
    """
    
    # YOUR CODE HERE
    pass

# Test cases for Exercise 1
test_case_1 = '''module test_module (
    input clk,
    input secret_input,
    output reg normal_output,
    output reg hidden_signal
);

always @(posedge clk) begin
    normal_output <= ~normal_output;
    if (secret_input == 1'b1)  // Potential backdoor
        hidden_signal <= 1'b1;
end

endmodule'''

print("Test your function with the provided test case!")
print("Expected: Should detect suspicious patterns on lines with secret_input logic")

# Exercise 2: Improve Security Scoring
print("\n🎯 EXERCISE 2: Enhance the security scoring algorithm")
print("=" * 60)

def exercise2_improved_scoring(verilog_code):
    """
    TODO: Improve the calculate_security_score function with:
    1. More sophisticated pattern detection
    2. Context-aware analysis
    3. Weighted risk assessment
    4. Machine learning integration
    """
    
    # HINT: Consider these factors:
    # - Conditional statements with specific values
    # - Unusual signal names
    # - Complex trigger conditions
    # - Data flow analysis
    
    # YOUR CODE HERE
    return 0.0  # Replace with your implementation

# Exercise 3: Create a Custom Classifier
print("\n🎯 EXERCISE 3: Implement a custom neural network classifier")
print("=" * 60)

class Exercise3TrojanClassifier(nn.Module):
    """
    TODO: Design a neural network for Trojan detection
    
    Requirements:
    1. Accept LLM embeddings as input
    2. Include multiple hidden layers
    3. Output binary classification (trojan/clean)
    4. Include dropout for regularization
    """
    
    def __init__(self, input_size=768, hidden_size=256):
        super().__init__()
        # YOUR CODE HERE
        # Example structure:
        # self.classifier = nn.Sequential(...)
        pass
    
    def forward(self, x):
        # YOUR CODE HERE
        pass

print("Design a neural network that can learn from LLM embeddings!")

# Exercise 4: Implement Advanced Visualization
print("\n🎯 EXERCISE 4: Create advanced Trojan visualization")
print("=" * 60)

def exercise4_advanced_viz(verilog_codes_list, titles_list):
    """
    TODO: Create a comprehensive comparison visualization
    
    Features to implement:
    1. Side-by-side code comparison
    2. Threat level heatmaps
    3. Temporal analysis (if applicable)
    4. Interactive filtering
    5. Export capabilities
    """
    
    # YOUR CODE HERE
    # Use plotly to create rich, interactive visualizations
    pass

print("Create visualizations that help security analysts understand threats!")

# Exercise Solutions and Hints
print("\n💡 HINTS FOR SUCCESS:")
print("=" * 30)
print("1. Study the existing functions for patterns and structure")
print("2. Use regular expressions for advanced pattern matching")
print("3. Consider both syntactic and semantic features")
print("4. Test your functions on various Verilog examples")
print("5. Document your approach and findings")

print("\n🏆 BONUS CHALLENGES:")
print("=" * 25)
print("1. Implement real-time Trojan detection")
print("2. Add support for SystemVerilog")
print("3. Create a web interface for your detector")
print("4. Integrate with existing EDA tools")
print("5. Develop countermeasures against detected Trojans")

# Grading rubric placeholder
print("\n📊 GRADING RUBRIC:")
print("=" * 20)
rubric = {
    "Exercise 1 - Detection Function": "25 points",
    "Exercise 2 - Security Scoring": "25 points", 
    "Exercise 3 - Custom Classifier": "25 points",
    "Exercise 4 - Visualization": "25 points",
    "Bonus Challenges": "Extra Credit"
}

for item, points in rubric.items():
    print(f"   {item}: {points}")

print("\n✅ Ready to start? Modify the template functions above!")

## 10. Summary and Real-World Applications

### 🎓 What You've Learned

Through this tutorial, you've gained hands-on experience with:

1. **Hardware Security Fundamentals**
   - Understanding Trojan threats in chip design
   - Recognizing different types of malicious insertions
   - Appreciating the importance of supply chain security

2. **LLM Applications in Hardware**
   - Using pre-trained language models for code analysis
   - Extracting semantic embeddings from Verilog code
   - Applying natural language processing to hardware design

3. **Machine Learning for Security**
   - Training classifiers for threat detection
   - Evaluating model performance on security tasks
   - Understanding the challenges of hardware security

4. **Practical Implementation**
   - Building end-to-end detection pipelines
   - Creating interactive visualizations
   - Developing tools for security analysts

### 🌟 Real-World Applications

TrojanLoc and similar technologies are being used in:

- **Semiconductor Companies**: Automated security verification
- **Government Agencies**: Supply chain integrity assurance  
- **Academic Research**: Advanced threat modeling
- **EDA Tool Integration**: Built-in security checking
- **Certification Bodies**: Compliance verification

### 🚀 Next Steps

To further your journey in LLM-based chip design:

1. **Explore Advanced Models**: Try domain-specific LLMs for hardware
2. **Scale Up**: Work with larger datasets and more complex designs
3. **Integrate Tools**: Connect with existing EDA workflows
4. **Research**: Investigate new threat models and detection methods
5. **Collaborate**: Join the hardware security community

### 📚 Additional Resources

- **Research Papers**: VeriLoc, TrojAI, Hardware Trojans survey papers
- **Datasets**: Trust-HUB, OpenROAD benchmarks
- **Tools**: Yosys, OpenLane, PyVerilog
- **Communities**: IEEE HOST, CHES, DAC security tracks

---

**🎉 Congratulations!** You've completed the TrojanLoc tutorial and are now equipped to apply LLMs to hardware security challenges!